# train

> One epoch of training, one epoch of validation, atomic checkpointing, and a resumable training loop with a phased loss schedule and differential learning rates.

The loop is generic: pass `num_classes` and it trains a binary or multi-class model with no other changes. Targets are integer-label `(B, H, W)` tensors throughout; the loss and metrics modules adapt internally.

In [ ]:
#| default_exp train

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn


In [ ]:
#| exporti
from segmentation_tools_pt.channels import DEFAULT_CHANNELS
from segmentation_tools_pt.losses import SegLoss, resolve_loss_weights
from segmentation_tools_pt.metrics import SegMetrics
from segmentation_tools_pt.model import get_param_groups, freeze_encoder, unfreeze_encoder
from segmentation_tools_pt.visualize import (
    save_color_dim_png, save_near_zero_png, save_overlay_grid_jpeg, save_multilabel_overlay_grid_jpeg,
)


## Training callbacks

Optional hooks at train / epoch / batch granularity. Subclass `TrainCallback` and pass a list to `train()` — the default loop is unchanged when `callbacks=None`.

Set `state.stop_training = True` in a callback (e.g. early stopping) to exit the epoch loop cleanly after the current epoch.

## One training epoch

One job: run the loader once, update the model, and return averaged losses and training metrics. Mixed precision is used when a `GradScaler` is supplied.

In [ ]:
#| export
@dataclass
class TrainState:
    """Mutable context passed to every training callback hook."""
    model: nn.Module
    device: torch.device
    num_epochs: int
    num_classes: int = 1
    epoch: int = 0
    batch_idx: int = 0
    optimizer: Optional[torch.optim.Optimizer] = None
    scheduler: Optional[Any] = None
    scaler: Optional[Any] = None
    criterion: Optional[SegLoss] = None
    train_loader: Any = None
    val_loader: Any = None
    batch: Optional[dict] = None
    logits: Optional[torch.Tensor] = None
    loss_dict: Optional[Dict[str, Any]] = None
    train_metrics: Optional[Dict[str, float]] = None
    val_metrics: Optional[Dict[str, float]] = None
    history: List[dict] = field(default_factory=list)
    best_score: float = 0.0
    best_metric: str = 'iou'
    checkpoint_dir: Optional[str] = None
    is_best: bool = False
    stop_training: bool = False


In [ ]:
#| export

class TrainCallback:
    """Base callback; override only the hooks you need. Lower `order` runs first."""
    order: int = 0

    def on_train_begin(self, state: TrainState) -> None: pass
    def on_train_end(self, state: TrainState) -> None: pass
    def on_epoch_begin(self, state: TrainState) -> None: pass
    def on_epoch_end(self, state: TrainState) -> None: pass
    def on_train_batch_begin(self, state: TrainState) -> None: pass
    def on_train_batch_end(self, state: TrainState) -> None: pass
    def on_val_begin(self, state: TrainState) -> None: pass
    def on_val_end(self, state: TrainState) -> None: pass
    def on_val_batch_begin(self, state: TrainState) -> None: pass
    def on_val_batch_end(self, state: TrainState) -> None: pass
    def on_checkpoint(self, state: TrainState, path: str) -> None: pass



## Default monitoring callbacks

fastai-style activation histograms (`ActivationStatsCallback`) and YOLO-style validation overlay JPEGs (`PredOverlayCallback`). Use `build_default_callbacks()` from Hydra or pass directly to `train()`.

In [ ]:
#| export
_HOOK_TYPES = (nn.Conv2d, nn.BatchNorm2d)

def _hookable_layers(model: nn.Module, max_layers: int) -> List[Tuple[str, nn.Module]]:
    "Up to `max_layers` hook targets, evenly spaced across Conv/BN modules."
    layers = [(n, m) for n, m in model.named_modules() if isinstance(m, _HOOK_TYPES) and n]
    if not layers:
        layers = [(n, m) for n, m in model.named_modules() if n]
    if not layers and isinstance(model, _HOOK_TYPES):
        # bare Conv2d/BatchNorm2d used directly as the whole model (e.g. tests) has no
        # named submodules at all -- named_modules() yields only ('', model).
        layers = [(type(model).__name__, model)]
    if len(layers) <= max_layers:
        return layers
    idxs = np.linspace(0, len(layers) - 1, max_layers, dtype=int)
    return [layers[i] for i in idxs]

class ActivationStatsCallback(TrainCallback):
    """Record per-layer activation histograms; save fastai colorful-dimension PNGs each epoch."""
    order = 10

    def __init__(
        self,
        checkpoint_dir: str,
        every: int = 1,
        n_bins: int = 40,
        hist_range: Tuple[float, float] = (0.0, 10.0),
        max_layers: int = 6,
    ):
        self.root = Path(checkpoint_dir) / 'activation_stats'
        self.every = every
        self.n_bins = n_bins
        self.hmin, self.hmax = hist_range
        self.max_layers = max_layers
        self._hooks: List[Any] = []
        self._layer_names: List[str] = []
        self._epoch_hists: List[List[torch.Tensor]] = []
        self._epoch_near_zero: List[List[float]] = []
        self._near_zero_history: Dict[str, List[Tuple[int, float]]] = {}
        self._pending: List[Optional[Tuple[torch.Tensor, float]]] = []

    def on_train_begin(self, state: TrainState) -> None:
        layers = _hookable_layers(state.model, self.max_layers)
        self._layer_names = [n.replace('.', '_') for n, _ in layers]
        self._epoch_hists = [[] for _ in layers]
        self._epoch_near_zero = [[] for _ in layers]
        self._near_zero_history = {name: [] for name in self._layer_names}
        self._pending = [None] * len(layers)

        def _make_hook(li: int):
            def hook(module, _inp, out):
                if not module.training:
                    return
                o = out.detach().float()
                if o.numel() == 0:
                    return
                flat = o.flatten()
                hist = torch.histc(flat, self.n_bins, self.hmin, self.hmax).cpu()
                nz = float((flat <= 0.05).float().mean().item())
                self._pending[li] = (hist, nz)
            return hook

        for i, (_, mod) in enumerate(layers):
            self._hooks.append(mod.register_forward_hook(_make_hook(i)))

    def on_train_batch_end(self, state: TrainState) -> None:
        if state.batch_idx % self.every != 0:
            return
        for i, item in enumerate(self._pending):
            if item is None:
                continue
            hist, nz = item
            self._epoch_hists[i].append(hist)
            self._epoch_near_zero[i].append(nz)
            self._pending[i] = None

    def on_epoch_end(self, state: TrainState) -> None:
        epoch_dir = self.root / f'epoch{state.epoch + 1:03d}'
        epoch_dir.mkdir(parents=True, exist_ok=True)
        for i, name in enumerate(self._layer_names):
            hists = self._epoch_hists[i]
            if not hists:
                continue
            stack = torch.stack(hists)
            save_color_dim_png(stack.T, epoch_dir / f'layer{i:02d}_{name}_color_dim.png', title=name)
            mean_nz = float(np.mean(self._epoch_near_zero[i]))
            self._near_zero_history[name].append((state.epoch, mean_nz))
            ep, vals = zip(*self._near_zero_history[name])
            save_near_zero_png(list(ep), list(vals), epoch_dir / f'layer{i:02d}_{name}_near_zero.png', title=name)
        self._epoch_hists = [[] for _ in self._layer_names]
        self._epoch_near_zero = [[] for _ in self._layer_names]

    def on_train_end(self, state: TrainState) -> None:
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

class PredOverlayCallback(TrainCallback):
    """Save YOLO-style validation overlay JPEG mosaics during training.

    `task='multilabel'` uses `visualize.save_multilabel_overlay_grid_jpeg`
    (per-channel colored GT-vs-Pred panels) instead of the single-label
    `save_overlay_grid_jpeg`.
    """
    order = 20

    def __init__(
        self,
        checkpoint_dir: str,
        threshold=0.5,
        every: int = 1,
        max_val_batches: int = 3,
        jpeg_quality: int = 90,
        num_classes: int = 1,
        task: str = None,   # 'binary' | 'multiclass' | 'multilabel'; None infers from num_classes
        class_names: Optional[Sequence[str]] = None,  # multilabel legend labels
    ):
        self.root = Path(checkpoint_dir) / 'predictions'
        self.threshold = threshold
        self.every = every
        self.max_val_batches = max_val_batches
        self.jpeg_quality = jpeg_quality
        self.num_classes = num_classes
        self.task = task or ('binary' if num_classes == 1 else 'multiclass')
        self.class_names = list(class_names) if class_names is not None else None

    def _decode_logits(self, logits: torch.Tensor) -> torch.Tensor:
        if self.num_classes == 1:
            return (torch.sigmoid(logits[:, 0]) >= self.threshold).long()
        return logits.argmax(dim=1)

    def on_val_batch_end(self, state: TrainState) -> None:
        if state.epoch % self.every != 0:
            return
        if state.batch_idx >= self.max_val_batches:
            return
        if state.batch is None or state.logits is None:
            return
        out_dir = self.root / f'epoch{state.epoch + 1:03d}'
        out_dir.mkdir(parents=True, exist_ok=True)
        images, masks = state.batch['image'], state.batch['mask']
        bi = state.batch_idx
        if self.task == 'multilabel':
            probs = torch.sigmoid(state.logits)
            save_multilabel_overlay_grid_jpeg(
                images, masks, probs,
                out_dir / f'val_batch{bi}_multilabel.jpg',
                class_names=self.class_names,
                threshold=self.threshold,
                jpeg_quality=self.jpeg_quality,
            )
            return
        preds = self._decode_logits(state.logits)
        for mode, suffix in (('labels', 'labels'), ('pred', 'pred'), ('compare', 'compare')):
            save_overlay_grid_jpeg(
                images, masks, preds,
                out_dir / f'val_batch{bi}_{suffix}.jpg',
                threshold=self.threshold,
                mode=mode,
                jpeg_quality=self.jpeg_quality,
            )

def build_default_callbacks(
    checkpoint_dir: str,
    threshold=0.5,
    activation_stats: bool = True,
    pred_overlays: bool = True,
    num_classes: int = 1,
    act_every: int = 1,
    act_n_bins: int = 40,
    act_hist_range: Tuple[float, float] = (0.0, 10.0),
    act_max_layers: int = 6,
    pred_every: int = 1,
    pred_max_val_batches: int = 3,
    pred_jpeg_quality: int = 90,
    task: str = None,   # 'binary' | 'multiclass' | 'multilabel'; None infers from num_classes
    class_names: Optional[Sequence[str]] = None,  # multilabel legend labels for PredOverlayCallback
) -> List[TrainCallback]:
    "Factory for on-by-default training monitoring callbacks."
    cbs: List[TrainCallback] = []
    if activation_stats:
        cbs.append(ActivationStatsCallback(
            checkpoint_dir, every=act_every, n_bins=act_n_bins,
            hist_range=act_hist_range, max_layers=act_max_layers,
        ))
    if pred_overlays:
        cbs.append(PredOverlayCallback(
            checkpoint_dir, threshold=threshold, every=pred_every,
            max_val_batches=pred_max_val_batches, jpeg_quality=pred_jpeg_quality,
            num_classes=num_classes, task=task, class_names=class_names,
        ))
    return cbs


In [ ]:
#| export
def _sorted_callbacks(callbacks: Optional[Sequence[TrainCallback]]) -> List[TrainCallback]:
    if not callbacks:
        return []
    return sorted(callbacks, key=lambda cb: getattr(cb, 'order', 0))


def _call_callbacks(
    callbacks: Optional[Sequence[TrainCallback]],
    hook: str,
    state: Optional[TrainState],
    **kwargs,
) -> None:
    if not callbacks or state is None:
        return
    for cb in _sorted_callbacks(callbacks):
        getattr(cb, hook)(state, **kwargs)


In [ ]:
#| export
def train_one_epoch(
    model: nn.Module,                  # model returning (B, C, H, W) logits
    loader,                            # training DataLoader of {'image','mask'} batches
    criterion: SegLoss,                # loss returning a breakdown dict
    optimizer: torch.optim.Optimizer,  # optimizer
    device: torch.device,              # cpu or cuda
    scaler: Optional[object] = None,   # torch.amp.GradScaler for AMP, or None
    grad_clip: float = 1.0,            # max gradient norm
    num_classes: int = 1,              # 1 = binary, >1 = multi-class or multilabel
    callbacks: Optional[Sequence[TrainCallback]] = None,
    state: Optional[TrainState] = None,
    task: str = None,                  # 'binary' | 'multiclass' | 'multilabel'; None infers from num_classes
    metrics_threshold=0.5,             # float, or per-channel Sequence[float] for task='multilabel'
) -> Dict[str, float]:
    """Train for one epoch; return averaged losses and `train_*` metrics."""
    model.train()
    running = {'loss': 0., 'loss_tversky': 0., 'loss_dice': 0., 'loss_boundary': 0.}
    metrics = SegMetrics(num_classes=num_classes, task=task, threshold=metrics_threshold)
    for batch_idx, batch in enumerate(loader):
        if state is not None:
            state.batch_idx = batch_idx
            state.batch = batch
        _call_callbacks(callbacks, 'on_train_batch_begin', state)
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['mask'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logits = model(images)
                ld = criterion(logits, targets)
            scaler.scale(ld['loss']).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(images)
            ld = criterion(logits, targets)
            ld['loss'].backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        if state is not None:
            state.logits = logits.detach()
            state.loss_dict = {k: float(v) for k, v in ld.items()}
        _call_callbacks(callbacks, 'on_train_batch_end', state)
        for k in running:
            running[k] += float(ld[k])
        metrics.update(logits.detach(), targets)
    n = len(loader)
    out = {k: v / n for k, v in running.items()}
    out.update({f'train_{k}': v for k, v in metrics.compute().items() if k != 'per_class'})
    return out


## Validation

In [ ]:
#| export
@torch.no_grad()
def validate(
    model: nn.Module,    # model returning logits
    loader,              # validation DataLoader
    criterion: SegLoss,  # loss for tracking val loss
    device: torch.device,
    num_classes: int = 1,
    callbacks: Optional[Sequence[TrainCallback]] = None,
    state: Optional[TrainState] = None,
    task: str = None,                  # 'binary' | 'multiclass' | 'multilabel'; None infers from num_classes
    metrics_threshold=0.5,             # float, or per-channel Sequence[float] for task='multilabel'
) -> Dict[str, float]:
    """Run validation; return overlap metrics plus `val_loss`."""
    model.eval()
    _call_callbacks(callbacks, 'on_val_begin', state)
    metrics = SegMetrics(num_classes=num_classes, task=task, threshold=metrics_threshold)
    running = 0.0
    for batch_idx, batch in enumerate(loader):
        if state is not None:
            state.batch_idx = batch_idx
            state.batch = batch
        _call_callbacks(callbacks, 'on_val_batch_begin', state)
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['mask'].to(device, non_blocking=True)
        logits = model(images)
        ld = criterion(logits, targets)
        running += float(ld['loss'])
        if state is not None:
            state.logits = logits
            state.loss_dict = {k: float(v) for k, v in ld.items()}
        _call_callbacks(callbacks, 'on_val_batch_end', state)
        metrics.update(logits, targets)
    res = {k: v for k, v in metrics.compute().items() if k != 'per_class'}
    res['val_loss'] = running / len(loader)
    if state is not None:
        state.val_metrics = res
    _call_callbacks(callbacks, 'on_val_end', state)
    return res


## Atomic checkpointing

Write to a temp file then rename, so an interrupted save never corrupts the last good checkpoint.

In [ ]:
#| export
def save_checkpoint(
    path,                       # destination .pth path
    epoch: int, model, optimizer, scheduler, scaler,  # training state
    best_score: float, history: list,                 # bookkeeping
) -> None:
    "Atomically save full training state."
    ckpt = dict(epoch=epoch, model_state=model.state_dict(), optimizer_state=optimizer.state_dict(),
                scheduler_state=scheduler.state_dict(),
                scaler_state=scaler.state_dict() if scaler else None,
                best_score=best_score, history=history)
    tmp = str(path) + '.tmp'
    torch.save(ckpt, tmp)
    Path(tmp).replace(path)

def load_checkpoint(path, model, optimizer, scheduler, scaler, device):
    "Restore training state; return `(start_epoch, best_score, history)`."
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    if scaler and ckpt.get('scaler_state'):
        scaler.load_state_dict(ckpt['scaler_state'])
    return ckpt['epoch'] + 1, ckpt['best_score'], ckpt['history']

## The training loop

Composes the pieces above: builds loaders, anneals the loss phase, optionally warms up with a frozen encoder, tracks the best validation mean-IoU, and writes resumable checkpoints.

In [ ]:
#| export
def train(
    root: str,                      # patch dataset root (None when pre-built loaders are supplied)
    model: nn.Module,               # model to train
    train_loader=None,              # pre-built DataLoader (direct mode); built from root if None
    val_loader=None,                # pre-built DataLoader (direct mode); built from root if None
    num_classes: int = 1,           # 1 = binary, >1 = multi-class or multilabel
    num_epochs: int = 60,           # total epochs
    batch_size: int = 16,           # batch size
    encoder_lr: float = 1e-4,       # encoder LR
    decoder_lr: float = 1e-3,       # decoder LR
    freeze_epochs: int = 0,         # epochs to keep the encoder frozen
    crop_size: int = 256,           # crop size
    n_source_channels: int = 2,     # raw channels per sample
    channel_names: Sequence[str] = DEFAULT_CHANNELS,  # model-image assembly
    mean: Optional[Sequence[float]] = None,  # per-channel mean
    std: Optional[Sequence[float]] = None,   # per-channel std
    device_str: str = 'cuda',       # device
    checkpoint_dir: str = 'checkpoints',  # where to save
    use_amp: bool = True,           # mixed precision on cuda
    save_every: int = 5,            # periodic checkpoint cadence
    resume: bool = True,            # resume from latest.pth if present
    loss_alpha: float = 0.7,        # Tversky FN weight (higher → fewer misses)
    loss_beta: float = 0.3,         # Tversky FP weight
    loss_gamma: float = 1.33,       # focal exponent
    loss_phases: Optional[list] = None,  # [{until_epoch, w_tversky, w_dice, w_boundary}]; None = fixed weights
    metrics_threshold=0.5,          # sigmoid threshold: float, or per-channel Sequence[float] for task='multilabel'
    best_metric: str = 'iou',       # metric to gate best.pth: iou | dice | recall | precision
    aug=None,                       # AugmentConfig for training augmentations
    boundary_blur: Optional[dict] = None,  # kwargs for BlurBoundaryZone (patch-mode training transform)
    num_workers: int = 4,           # DataLoader worker processes
    resume_from: Optional[str] = None,  # explicit checkpoint path to resume from; None = use latest.pth
    seed: Optional[int] = None,     # run seed for reproducible sampling and worker RNGs
    callbacks: Optional[Sequence[TrainCallback]] = None,
    task: str = None,                              # 'binary' | 'multiclass' | 'multilabel'; None infers from num_classes
    mask_mode: str = 'single',                     # 'single' | 'stacked' | 'per_class_files' (patch-mode dataset only)
    mask_channels: Optional[int] = None,           # K, required for mask_mode='stacked'
    class_names: Optional[Sequence[str]] = None,   # K class name suffixes, required for mask_mode='per_class_files'
) -> list:
    """Resumable training loop with configurable loss phases; returns per-epoch history."""
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import CosineAnnealingLR
    from torch.utils.data import DataLoader, WeightedRandomSampler
    from segmentation_tools_pt.dataset import SegmentationDataset, _make_loader_generator, _seed_worker
    from segmentation_tools_pt.transforms import get_train_transforms, get_val_transforms

    Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    device = torch.device(device_str if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    # Build loaders from patch root when not supplied (patch mode)
    if train_loader is None or val_loader is None:
        c = len(channel_names)
        _mean = tuple(mean) if mean is not None else (0.,) * c
        _std  = tuple(std)  if std  is not None else (1.,) * c
        common = dict(num_classes=num_classes, n_source_channels=n_source_channels,
                      channel_names=channel_names,
                      mask_mode=mask_mode, mask_channels=mask_channels, class_names=class_names)
        train_ds = SegmentationDataset(
            root, 'train',
            transform=get_train_transforms(crop_size, _mean, _std, aug=aug, boundary_blur=boundary_blur),
            **common,
        )
        val_ds = SegmentationDataset(
            root, 'val',
            transform=get_val_transforms(crop_size, _mean, _std),
            **common,
        )
        sampler = WeightedRandomSampler(
            weights=train_ds.get_sample_weights(),
            num_samples=len(train_ds),
            replacement=True,
            generator=_make_loader_generator(seed),
        )
        pin = (device.type == 'cuda')
        worker_init_fn = _seed_worker if num_workers > 0 else None
        train_loader = DataLoader(
            train_ds, batch_size=batch_size, sampler=sampler,
            num_workers=num_workers, pin_memory=pin, drop_last=True,
            persistent_workers=(num_workers > 0),
            worker_init_fn=worker_init_fn,
            generator=_make_loader_generator(seed),
        )
        val_loader = DataLoader(
            val_ds, batch_size=batch_size, shuffle=False,
            num_workers=num_workers, pin_memory=pin, drop_last=False,
            persistent_workers=(num_workers > 0),
            worker_init_fn=worker_init_fn,
            generator=_make_loader_generator(None if seed is None else seed + 1),
        )

    # Configure loss — initial weights from first phase
    _phases = loss_phases or [{'until_epoch': -1, 'w_tversky': 0.6, 'w_dice': 0.4, 'w_boundary': 0.0}]
    _w0 = resolve_loss_weights(0, _phases)
    criterion = SegLoss(alpha=loss_alpha, beta=loss_beta, gamma=loss_gamma,
                        w_tversky=_w0[0], w_dice=_w0[1], w_boundary=_w0[2], task=task).to(device)
    optimizer = AdamW(get_param_groups(model, encoder_lr, decoder_lr), weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda') if (use_amp and device.type == 'cuda') else None

    start_epoch, best_score, history = 0, 0.0, []
    latest = Path(checkpoint_dir) / 'latest.pth'
    if resume:
        if resume_from is not None:
            _resume_path = Path(resume_from)
        else:
            _resume_path = latest
        if _resume_path.exists():
            print(f'Resuming from {_resume_path}')
            start_epoch, best_score, history = load_checkpoint(str(_resume_path), model, optimizer, scheduler, scaler, device)
        elif resume_from is not None:
            raise FileNotFoundError(f'resume_from checkpoint not found: {_resume_path}')
    if freeze_epochs > 0:
        freeze_encoder(model)

    state = TrainState(
        model=model, device=device, num_epochs=num_epochs, num_classes=num_classes,
        optimizer=optimizer, scheduler=scheduler, scaler=scaler, criterion=criterion,
        train_loader=train_loader, val_loader=val_loader,
        history=history, best_score=best_score, best_metric=best_metric,
        checkpoint_dir=checkpoint_dir,
    )
    _call_callbacks(callbacks, 'on_train_begin', state)

    for epoch in range(start_epoch, num_epochs):
        state.epoch = epoch
        state.history = history
        state.best_score = best_score
        _call_callbacks(callbacks, 'on_epoch_begin', state)
        _w = resolve_loss_weights(epoch, _phases)
        criterion.set_weights(*_w)
        if freeze_epochs > 0 and epoch == freeze_epochs:
            unfreeze_encoder(model)
        tr = train_one_epoch(
            model, train_loader, criterion, optimizer, device, scaler,
            num_classes=num_classes, callbacks=callbacks, state=state,
            task=task, metrics_threshold=metrics_threshold,
        )
        state.train_metrics = tr
        va = validate(
            model, val_loader, criterion, device, num_classes=num_classes,
            callbacks=callbacks, state=state,
            task=task, metrics_threshold=metrics_threshold,
        )
        scheduler.step()
        log = dict(epoch=epoch, train_loss=tr['loss'], val_loss=va['val_loss'],
                   val_iou=va['iou'], val_dice=va['dice'], val_recall=va['recall'],
                   lr=scheduler.get_last_lr()[0])
        history.append(log)
        (Path(checkpoint_dir) / 'history.json').write_text(json.dumps(history, indent=2))
        args = (epoch, model, optimizer, scheduler, scaler)
        is_best = va[best_metric] > best_score
        state.is_best = is_best
        print(
            f"epoch {epoch+1:03d}/{num_epochs}"
            f"  train={tr['loss']:.4f}"
            f"  val={va['val_loss']:.4f}"
            f"  iou={va['iou']:.4f}"
            f"  dice={va['dice']:.4f}"
            f"  recall={va['recall']:.4f}"
            f"  lr={scheduler.get_last_lr()[0]:.2e}"
            + ("  *best*" if is_best else ""),
            flush=True,
        )
        if is_best:
            best_score = va[best_metric]
            state.best_score = best_score
            best_path = str(Path(checkpoint_dir) / 'best.pth')
            save_checkpoint(best_path, *args, best_score, history)
            _call_callbacks(callbacks, 'on_checkpoint', state, path=best_path)
        if (epoch + 1) % save_every == 0:
            periodic_path = str(Path(checkpoint_dir) / f'periodic_ep{epoch:03d}.pth')
            save_checkpoint(periodic_path, *args, best_score, history)
            _call_callbacks(callbacks, 'on_checkpoint', state, path=periodic_path)
        latest_path = str(latest)
        save_checkpoint(latest_path, *args, best_score, history)
        _call_callbacks(callbacks, 'on_checkpoint', state, path=latest_path)
        _call_callbacks(callbacks, 'on_epoch_end', state)
        if state.stop_training:
            break
    _call_callbacks(callbacks, 'on_train_end', state)
    return history

In [ ]:
# smoke test: one epoch + validate + checkpoint round-trip + monitoring callbacks
import tempfile
from torch.utils.data import Dataset, DataLoader

class _Toy(Dataset):
    def __len__(self): return 8
    def __getitem__(self, i):
        return {'image': torch.rand(3, 16, 16), 'mask': (torch.rand(16, 16) > 0.5).long(), 'sample_id': str(i)}

class _CountBatches(TrainCallback):
    def __init__(self): self.train_batches = self.val_batches = 0
    def on_train_batch_end(self, state): self.train_batches += 1
    def on_val_batch_end(self, state): self.val_batches += 1

loader = DataLoader(_Toy(), batch_size=2)
model = nn.Conv2d(3, 1, 3, padding=1)
crit = SegLoss(w_tversky=0.5, w_dice=0.5, w_boundary=0.0)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
dev = torch.device('cpu')
st = TrainState(model=model, device=dev, num_epochs=1)
cb = _CountBatches()
tr = train_one_epoch(model, loader, crit, opt, dev, num_classes=1, callbacks=[cb], state=st)
assert 'loss' in tr and 'train_iou' in tr
assert cb.train_batches == len(loader)
va = validate(model, loader, crit, dev, num_classes=1, callbacks=[cb], state=st)
assert 'val_loss' in va and 'iou' in va
assert cb.val_batches == len(loader)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=2)
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / 'ckpt.pth'
    save_checkpoint(p, 0, model, opt, sched, None, 0.0, [])
    se, best, hist = load_checkpoint(p, model, opt, sched, None, dev)
    assert se == 1 and best == 0.0
    ckpt = Path(d) / 'checkpoints'
    mon = build_default_callbacks(str(ckpt), threshold=0.5, num_classes=1)
    hist2 = train(
        root=None, model=model, train_loader=loader, val_loader=loader,
        num_epochs=1, batch_size=2, device_str='cpu', checkpoint_dir=str(ckpt),
        use_amp=False, resume=False, callbacks=mon,
    )
    assert len(hist2) == 1
    assert list((ckpt / 'activation_stats').rglob('*_color_dim.png'))
    assert list((ckpt / 'predictions').rglob('val_batch0_labels.jpg'))


In [ ]:
# smoke test: multilabel (K=3) train loop end to end -- loss, metrics, and per-channel overlays all task-aware
class _ToyMultilabel(Dataset):
    def __len__(self): return 8
    def __getitem__(self, i):
        return {'image': torch.rand(3, 16, 16), 'mask': (torch.rand(3, 16, 16) > 0.5).float(), 'sample_id': str(i)}

ml_loader = DataLoader(_ToyMultilabel(), batch_size=2)
ml_model = nn.Conv2d(3, 3, 3, padding=1)
with tempfile.TemporaryDirectory() as d:
    ckpt = Path(d) / 'checkpoints'
    mon = build_default_callbacks(str(ckpt), num_classes=3, task='multilabel', pred_overlays=True,
                                   class_names=['chip', 'void', 'lifted'])
    hist = train(
        root=None, model=ml_model, train_loader=ml_loader, val_loader=ml_loader,
        num_classes=3, num_epochs=1, batch_size=2, device_str='cpu', checkpoint_dir=str(ckpt),
        use_amp=False, resume=False, task='multilabel', metrics_threshold=[0.5, 0.5, 0.5],
        callbacks=mon,
    )
    assert len(hist) == 1
    # PredOverlayCallback now renders per-channel colored GT-vs-Pred overlays for multilabel
    ml_jpgs = list((ckpt / 'predictions').rglob('*_multilabel.jpg'))
    assert ml_jpgs, 'expected multilabel overlay JPEGs'
    assert not list((ckpt / 'predictions').rglob('*_compare.jpg'))  # single-label filenames not used
    # ActivationStatsCallback has no task dependency, still runs
    assert list((ckpt / 'activation_stats').rglob('*_color_dim.png'))
print('train() ✓  multilabel: task threaded through loss/metrics, per-channel overlays rendered')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()